# Per-Invocation

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_ollama import ChatOllama
from rich import print
model = ChatOllama(model="minimax-m3:cloud", temperature=0.9)

@tool
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    return f"Info about {fruit_name}"

@tool
def veggie_info(veggie_name: str) -> str:
    """Look up veggie info."""
    return f"Info about {veggie_name}"

# Subagents - Updated 'prompt' to 'system_prompt'
fruit_agent = create_agent(
    model=model,
    tools=[fruit_info],
    system_prompt="You are a fruit expert. Use the fruit_info tool. Respond in one sentence.",
)

veggie_agent = create_agent(
    model=model,
    tools=[veggie_info],
    system_prompt="You are a veggie expert. Use the veggie_info tool. Respond in one sentence.",
)

# Wrap subagents as tools for the outer agent
@tool
def ask_fruit_expert(question: str) -> str:
    """Ask the fruit expert. Use for ALL fruit questions."""
    response = fruit_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content

@tool
def ask_veggie_expert(question: str) -> str:
    """Ask the veggie expert. Use for ALL veggie questions."""
    response = veggie_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content

# Outer agent - Updated 'prompt' to 'system_prompt'
agent = create_agent(
    model=model,
    tools=[ask_fruit_expert, ask_veggie_expert],
    system_prompt=(
        "You have two experts: ask_fruit_expert and ask_veggie_expert. "
        "ALWAYS delegate questions to the appropriate expert."
    ),
    checkpointer=MemorySaver(),
)
config = {"configurable": {"thread_id": "session-123"}}
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about an apple."}]}, 
    config=config
)
print(response["messages"][-1].content)


Here's what our fruit expert shared about apples:

Apples are a popular and versatile fruit enjoyed fresh, baked, or juiced. They come in many varieties like 
**Gala**, **Fuji**, **Granny Smith**, and **Honeycrisp**, each offering unique flavors and textures.

**Nutritional benefits:**
- Rich in fiber
- High in vitamin C
- Packed with antioxidants
- Supports heart and digestive health

**How to pick a good apple:**
- Look for firm, smooth skin
- Choose vibrant color appropriate to the variety
- Smell for a fresh fragrance
- Pick ones that feel heavy for their size (a sign of juiciness!)

Let me know if you'd like to know more — for example, I can also ask the veggie expert about other produce or the 
fruit expert about specific apple recipes or storage tips.

# Per-Thread

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_ollama import ChatOllama
from rich import print

# 1. Initialize your model instance explicitly
model = ChatOllama(model="minimax-m3:cloud", temperature=0.9)

@tool
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    return f"Info about {fruit_name}"

# 2. Subagent Configuration - Changed 'prompt' to 'system_prompt'
fruit_agent = create_agent(
    model=model,
    tools=[fruit_info],
    system_prompt="You are a fruit expert. Use the fruit_info tool. Respond in one sentence.",
    checkpointer=True,
)

# 3. Isolated tool execution wrapper
@tool
def ask_fruit_expert(question: str) -> str:
    """Ask the fruit expert. Use for ALL fruit questions."""
    response = fruit_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content

# 4. Outer Agent Configuration with Middleware constraints
agent = create_agent(
    model=model,
    tools=[ask_fruit_expert],
    system_prompt="You have a fruit expert. ALWAYS delegate fruit questions to ask_fruit_expert.",
    middleware=[
        ToolCallLimitMiddleware(tool_name="ask_fruit_expert", run_limit=1),
    ],
    checkpointer=MemorySaver(),
)

# 5. Stateful execution using LangGraph thread configurations
config = {"configurable": {"thread_id": "session-123"}}
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about an apple."}]}, 
    config=config
)

print(response["messages"][-1].content)


**Apples** are a classic and widely enjoyed fruit from the genus *Malus* (the domestic apple is *Malus domestica*).
Here are some key facts:

- **Appearance**: They typically have round shape with skin that can be red, green, yellow, or a combination of 
these colors.
- **Taste**: The flavor ranges from sweet to tart, depending on the variety (e.g., Fuji, Granny Smith, Honeycrisp, 
Gala).
- **Texture**: Known for being crisp and juicy when fresh.
- **Nutrition**: A good source of fiber (especially pectin), vitamin C, and various antioxidants. "An apple a 
day..." as the saying goes!
- **Uses**: Eaten fresh, baked (think apple pie), pressed into cider or juice, dried, or made into sauces and 
butters.
- **Fun facts**: There are over 7,500 known varieties worldwide, and apples float because they're about 25% air.
- **Cultural significance**: Apples appear in mythology, religion, and literature (e.g., the forbidden fruit in 
some traditions, Newton's apple, "Snow White," etc.).

Would you like to know about a specific variety, how to pick a good one, or something else?